In [ ]:
%load_ext autoreload
%autoreload 2

from math import pi
from glob import glob

import numpy as np
import matplotlib.pyplot as plt

import ast

import pandas as pd

In [ ]:
from matplotlib.colors import LogNorm

In [ ]:
plt.rcParams.update({'font.size': 18})

In [ ]:
import scipy.signal

In [ ]:
from numpy.fft import fft, ifft, ifftshift

# Load data

In [ ]:
wfdf = pd.read_csv('pmt_waveforms.csv',\
                   converters={'wf_00': ast.literal_eval,'wf_01': ast.literal_eval,'wf_02': ast.literal_eval})

In [ ]:
print (wfdf.shape)

# Deconvolve some waveforms

### load kernel, SNR, and noise in freq. space

In [ ]:
fin = open('spekernelpmt00.txt','r')

kernel00 = []
for line in fin:
    words = line.split(',')
    for word in words:
        try:
            val = float(word)
            #print(val)
            if (isinstance(val,float)):
                kernel00.append(val)
        except ValueError:
            continue
    kernel00 = np.array(kernel00)
    #print(kernel00)
    break
    
fin = open('spenoisemag00.txt','r')
noise00 = []
for line in fin:
    words = line.split(',')
    for word in words:
        try:
            val = float(word)
            if (isinstance(val,float)):
                noise00.append(val)
        except ValueError:
            continue
    noise00 = np.array(noise00)
    break
    
fin = open('snrmag00.txt','r')
snr00 = []
for line in fin:
    words = line.split(',')
    for word in words:
        try:
            val = float(word)
            if (isinstance(val,float)):
                snr00.append(val)
        except ValueError:
            continue
    snr00 = np.array(snr00)
    break

In [ ]:
fig = plt.figure(figsize=(10,4))
plt.plot(kernel00,'ro-')
plt.xlabel('time-tick [64 MHz]')
plt.ylabel('Amplitude')
#plt.ylim([-0.2,0.2])
plt.show()

In [ ]:
fig = plt.figure(figsize=(10,4))
plt.plot(snr00,'ro-')
plt.xlabel('Freq. [MHz]')
plt.ylabel('Amplitude')
plt.yscale('log')
#plt.ylim([-0.2,0.2])
plt.show()
print(len(snr00))

In [ ]:
# https://gist.github.com/danstowell/f2d81a897df9e23cc1da
def wiener_deconvolution(signal, kernel, lambd):
    #"lambd is the SNR"
    kernel = np.hstack((kernel, np.zeros(len(signal) - len(kernel)))) # zero pad the kernel to same length
    H = fft(kernel)
    deconvolved = np.real(ifft(fft(signal)*np.conj(H)/(H*np.conj(H) + lambd**2)))
    return deconvolved

# https://gist.github.com/danstowell/f2d81a897df9e23cc1da
def wiener_deconvolution_SNR(signal, kernel, SNR):
    #"lambd is the SNR"
    kernel = np.hstack((kernel, np.zeros(len(signal) - len(kernel)))) # zero pad the kernel to same length
    buffer = np.ones(len(signal)-len(SNR))
    buffer *= 1e-2
    SNRpad = np.hstack((SNR, buffer))
    H = fft(kernel)
    G = (1/H)*(1./(1+1./((H*np.conj(H))*SNRpad)))
    deconvolved = np.real(ifft(fft(signal)*G))
    return deconvolved

In [ ]:
kernel00norm = kernel00/np.max(kernel00)

In [ ]:
ctr = 0
for idx,row in wfdf.iterrows():
    
    wf00 = np.array(row['wf_00'])-2048
    
    #if(np.max(wf00) < 300): continue
    
    deconvc = wiener_deconvolution(wf00[0:1500], kernel00norm, lambd=1)
    deconv = wiener_deconvolution_SNR(wf00[0:1500], kernel00norm, snr00)
    
    print(deconv)
    
    deconv = np.array(deconv)
    deconv /= np.max(deconv)

    deconvc = np.array(deconvc)
    deconvc /= np.max(deconvc)
    
    wf00 = wf00/np.max(wf00)
    
    
    fig = plt.figure(figsize=(15,6))
    
    plt.plot(wf00,'k--',label='original wf')
    plt.plot(deconv,'b--',label='deconvol. SNR')
    plt.plot(deconvc,'r--',label=r'deconvol. $\lambda$')
    
    plt.xlabel('Time Tick [15.625 ns / 64 MHz]',fontsize=16)
    plt.ylabel('ADC',fontsize=16)
    plt.legend(loc=1,fontsize=18)
    plt.xlim([1000,1400])
    plt.title(r'23.4 $\mu$s unbiased MicroBooNE PMT readout waveform')
    plt.tight_layout()
    plt.show()
    ctr += 1
    if (ctr >= 10):
        break